In [1]:
# Block 1: Setup and Data Loading

# Import necessary libraries
import pandas as pd
import numpy as np
from pulp import LpProblem, LpVariable, LpMinimize, lpSum, value
import matplotlib.pyplot as plt
import os
from PIL import Image

# 1. Define the path to your file
file_name = 'Nurse_fulldata.csv'

# 2. VERIFIED Column Map
column_map = {
    'PROVNUM': 'PROVNUM',
    'WorkDate': 'Date',
    'MDScensus': 'MDScensus',
    'Hrs_RN_emp': 'RN_employee_hrs',
    'Hrs_LPN_emp': 'LPN_employee_hrs',
    'Hrs_CNA_emp': 'CNA_employee_hrs',
    'Hrs_RN_ctr': 'RN_contract_hrs',
    'Hrs_LPN_ctr': 'LPN_contract_hrs',
    'Hrs_CNA_ctr': 'CNA_contract_hrs'
}

# --- Code to Load and Prepare Your Data ---
try:
    # low_memory=False handles large files; encoding='latin1' handles special characters
    df = pd.read_csv(file_name, encoding='latin1', low_memory=False)
    print(f"Successfully loaded {file_name}. Original shape: {df.shape}")
except Exception as e:
    print(f"An error occurred during file loading: {e}")
    raise

# Apply the column mapping
df.rename(columns=column_map, inplace=True)
print("Columns successfully renamed.")
print("\nFirst 5 rows of the loaded and renamed data:")
print(df.head())

Successfully loaded Nurse_fulldata.csv. Original shape: (1309590, 33)
Columns successfully renamed.

First 5 rows of the loaded and renamed data:
  PROVNUM                  PROVNAME          CITY STATE COUNTY_NAME  \
0  015009  BURNS NURSING HOME, INC.  RUSSELLVILLE    AL    Franklin   
1  015009  BURNS NURSING HOME, INC.  RUSSELLVILLE    AL    Franklin   
2  015009  BURNS NURSING HOME, INC.  RUSSELLVILLE    AL    Franklin   
3  015009  BURNS NURSING HOME, INC.  RUSSELLVILLE    AL    Franklin   
4  015009  BURNS NURSING HOME, INC.  RUSSELLVILLE    AL    Franklin   

   COUNTY_FIPS  CY_Qtr      Date  MDScensus  Hrs_RNDON  ...  LPN_contract_hrs  \
0           59  2025Q1  20250101         51       0.00  ...               0.0   
1           59  2025Q1  20250102         48      10.12  ...               0.0   
2           59  2025Q1  20250103         48       9.65  ...               0.0   
3           59  2025Q1  20250104         47       0.00  ...               0.0   
4           59  2025Q1

In [2]:
# Block 2: Data Cleaning and Aggregation

# --- Data Cleaning & Aggregation ---

# Convert hour and census columns to numeric, coercing errors to NaN
numeric_cols = ['MDScensus', 'RN_employee_hrs', 'LPN_employee_hrs', 'CNA_employee_hrs',
                'RN_contract_hrs', 'LPN_contract_hrs', 'CNA_contract_hrs']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop unusable rows (missing facility ID or census) and filter out days with zero census
df.dropna(subset=['PROVNUM', 'MDScensus'], inplace=True)
df_cleaned = df[df['MDScensus'] > 0].copy()

# Fill any remaining blank hour values with 0 (assuming missing hours = zero hours worked)
hour_cols = ['RN_employee_hrs', 'LPN_employee_hrs', 'CNA_employee_hrs',
             'RN_contract_hrs', 'LPN_contract_hrs', 'CNA_contract_hrs']
df_cleaned[hour_cols] = df_cleaned[hour_cols].fillna(0)

# Aggregate daily data by facility (PROVNUM)
facility_summary = df_cleaned.groupby('PROVNUM').agg(
    Total_Resident_Days=pd.NamedAgg(column='MDScensus', aggfunc='sum'),
    Total_RN_Hours=pd.NamedAgg(column='RN_employee_hrs', aggfunc='sum'),
    Total_LPN_Hours=pd.NamedAgg(column='LPN_employee_hrs', aggfunc='sum'),
    Total_CNA_Hours=pd.NamedAgg(column='CNA_employee_hrs', aggfunc='sum'),
    Total_RN_Contract_Hours=pd.NamedAgg(column='RN_contract_hrs', aggfunc='sum'),
    Total_LPN_Contract_Hours=pd.NamedAgg(column='LPN_contract_hrs', aggfunc='sum'),
    Total_CNA_Contract_Hours=pd.NamedAgg(column='CNA_contract_hrs', aggfunc='sum')
).reset_index()

# Calculate total original hours (employee + contract)
facility_summary['Original_RN_Hours'] = facility_summary['Total_RN_Hours'] + facility_summary['Total_RN_Contract_Hours']
facility_summary['Original_LPN_Hours'] = facility_summary['Total_LPN_Hours'] + facility_summary['Total_LPN_Contract_Hours']
facility_summary['Original_CNA_Hours'] = facility_summary['Total_CNA_Hours'] + facility_summary['Total_CNA_Contract_Hours']
facility_summary['Original_Total_Hours'] = facility_summary['Original_RN_Hours'] + facility_summary['Original_LPN_Hours'] + facility_summary['Original_CNA_Hours']

print("Aggregated Facility Data Head:")
print(facility_summary.head())
print(f"\nNumber of facilities to analyze: {len(facility_summary)}")

Aggregated Facility Data Head:
  PROVNUM  Total_Resident_Days  Total_RN_Hours  Total_LPN_Hours  \
0  015009                 4010         3646.33          2082.19   
1  015010                 6765         2645.50          6107.25   
2  015012                 3884         2344.38          3459.11   
3  015014                 2224         1147.16          2040.16   
4  015015                 8188         3040.76          5896.41   

   Total_CNA_Hours  Total_RN_Contract_Hours  Total_LPN_Contract_Hours  \
0         11388.64                      0.0                       0.0   
1         15483.50                      0.0                       0.0   
2         10762.88                      0.0                       0.0   
3          6010.50                      0.0                       0.0   
4         18523.72                      0.0                       0.0   

   Total_CNA_Contract_Hours  Original_RN_Hours  Original_LPN_Hours  \
0                       0.0            3646.33           

In [3]:
# Block 3: The Optimization Model (PuLP)

# --- Optimization ---
results = []

# Define the minimum HPRD requirements
HPRD_MIN = {'RN': 0.75, 'LPN': 0.55, 'CNA': 2.25}

print(f"Starting optimization for {len(facility_summary)} facilities...")

# Loop through each facility to run the optimization model
for index, row in facility_summary.iterrows():
    facility_id = row['PROVNUM']
    total_resident_days = row['Total_Resident_Days']
    
    # Skip facilities with no resident days
    if total_resident_days <= 0: continue

    # 1. Define the Problem (Objective: Minimize total hours)
    model = LpProblem(name=f"Nurse_Staffing_{facility_id}", sense=LpMinimize)
    
    # 2. Define Variables (The hours to be determined by the model)
    XfRN_emp = LpVariable(name=f"XfRN_emp_{facility_id}", lowBound=0, cat='Continuous')
    XfRN_ctr = LpVariable(name=f"XfRN_ctr_{facility_id}", lowBound=0, cat='Continuous')
    XfLPN_emp = LpVariable(name=f"XfLPN_emp_{facility_id}", lowBound=0, cat='Continuous')
    XfLPN_ctr = LpVariable(name=f"XfLPN_ctr_{facility_id}", lowBound=0, cat='Continuous')
    XfCNA_emp = LpVariable(name=f"XfCNA_emp_{facility_id}", lowBound=0, cat='Continuous')
    XfCNA_ctr = LpVariable(name=f"XfCNA_ctr_{facility_id}", lowBound=0, cat='Continuous')
    
    # Objective function: Minimize the sum of all staff hours
    objective = lpSum([XfRN_emp, XfRN_ctr, XfLPN_emp, XfLPN_ctr, XfCNA_emp, XfCNA_ctr])
    model += objective
    
    # 3. Define Constraints (HPRD minimums for each nurse type)
    # Total RN Hours (Employee + Contract) >= Required RN Hours
    model += (XfRN_emp + XfRN_ctr) >= HPRD_MIN['RN'] * total_resident_days, "RN_HPRD_Requirement"
    # Total LPN Hours >= Required LPN Hours
    model += (XfLPN_emp + XfLPN_ctr) >= HPRD_MIN['LPN'] * total_resident_days, "LPN_HPRD_Requirement"
    # Total CNA Hours >= Required CNA Hours
    model += (XfCNA_emp + XfCNA_ctr) >= HPRD_MIN['CNA'] * total_resident_days, "CNA_HPRD_Requirement"
    
    # 4. Solve the model
    model.solve()
    
    # 5. Store the results
    results.append({
        'PROVNUM': facility_id,
        'Optimized_RN_Hours': value(XfRN_emp) + value(XfRN_ctr),
        'Optimized_LPN_Hours': value(XfLPN_emp) + value(XfLPN_ctr),
        'Optimized_CNA_Hours': value(XfCNA_emp) + value(XfCNA_ctr),
        'Optimized_Total_Hours': value(model.objective)
    })

results_df = pd.DataFrame(results)
print("Optimization complete.")
print("\nOptimization Results Head (Total Optimized Hours per nurse type):")
print(results_df.head())

Starting optimization for 14551 facilities...
Optimization complete.

Optimization Results Head (Total Optimized Hours per nurse type):
  PROVNUM  Optimized_RN_Hours  Optimized_LPN_Hours  Optimized_CNA_Hours  \
0  015009             3007.50              2205.50              9022.50   
1  015010             5073.75              3720.75             15221.25   
2  015012             2913.00              2136.20              8739.00   
3  015014             1668.00              1223.20              5004.00   
4  015015             6141.00              4503.40             18423.00   

   Optimized_Total_Hours  
0               14235.50  
1               24015.75  
2               13788.20  
3                7895.20  
4               29067.40  


In [4]:
# Block 4: Evaluation, Compliance, and Staffing Status

# --- Evaluation ---
final_df = pd.merge(facility_summary, results_df, on='PROVNUM')

# Calculate the difference between original and optimized hours
final_df['Hours_Saved'] = final_df['Original_Total_Hours'] - final_df['Optimized_Total_Hours']

print("\nFinal Comparison Data Head (Original vs. Optimized Hours):")
print(final_df[['PROVNUM', 'Original_Total_Hours', 'Optimized_Total_Hours', 'Hours_Saved']].head())

# --- Compliance and Savings Calculation ---
# Calculate Original HPRD
final_df['Original_RN_HPRD'] = final_df['Original_RN_Hours'] / final_df['Total_Resident_Days']
final_df['Original_LPN_HPRD'] = final_df['Original_LPN_Hours'] / final_df['Total_Resident_Days']
final_df['Original_CNA_HPRD'] = final_df['Original_CNA_Hours'] / final_df['Total_Resident_Days']

# Determine Original and Optimized Compliance (Optimization ensures 100% compliance)
final_df['Original_Compliance'] = ((final_df['Original_RN_HPRD'] >= HPRD_MIN['RN']) & 
                                   (final_df['Original_LPN_HPRD'] >= HPRD_MIN['LPN']) & 
                                   (final_df['Original_CNA_HPRD'] >= HPRD_MIN['CNA']))

original_compliance_pct = final_df['Original_Compliance'].mean() * 100
optimized_compliance_pct = 100 # Optimization is mathematically designed to hit compliance

total_hours_saved = final_df['Hours_Saved'].sum()
total_original_hours = final_df['Original_Total_Hours'].sum()
percent_hours_saved = (total_hours_saved / total_original_hours) * 100 if total_original_hours > 0 else 0


print("\n--- EVALUATION SUMMARY ---")
print(f"Original Compliance Rate: {original_compliance_pct:.2f}% of facilities were compliant.")
print(f"Optimized Compliance Rate: {optimized_compliance_pct:.2f}% of facilities are now compliant.")
print(f"Total Hours Before Optimization: {total_original_hours:,.0f}")
print(f"Total Hours After Optimization: {final_df['Optimized_Total_Hours'].sum():,.0f}")
print(f"Total Hours Saved (Net Change): {total_hours_saved:,.0f}")
print(f"Percentage of Hours Saved (Net Change): {percent_hours_saved:.2f}%")


# --- Detailed Staffing Status Analysis ---
print("\n--- Detailed Staffing Status Analysis ---")
final_df['Required_RN_Hours'] = final_df['Total_Resident_Days'] * HPRD_MIN['RN']
final_df['Required_LPN_Hours'] = final_df['Total_Resident_Days'] * HPRD_MIN['LPN']
final_df['Required_CNA_Hours'] = final_df['Total_Resident_Days'] * HPRD_MIN['CNA']

choices = ['Understaffed', 'Overstaffed']
# RN Status
conditions_rn = [
    final_df['Original_RN_Hours'] < final_df['Required_RN_Hours'],
    final_df['Original_RN_Hours'] > final_df['Required_RN_Hours']
]
final_df['RN_Status'] = np.select(conditions_rn, choices, default='Optimal')

# LPN Status
conditions_lpn = [
    final_df['Original_LPN_Hours'] < final_df['Required_LPN_Hours'],
    final_df['Original_LPN_Hours'] > final_df['Required_LPN_Hours']
]
final_df['LPN_Status'] = np.select(conditions_lpn, choices, default='Optimal')

# CNA Status
conditions_cna = [
    final_df['Original_CNA_Hours'] < final_df['Required_CNA_Hours'],
    final_df['Original_CNA_Hours'] > final_df['Required_CNA_Hours']
]
final_df['CNA_Status'] = np.select(conditions_cna, choices, default='Optimal')

print("\nDetailed status for each facility (first 5 rows):")
print(final_df[['PROVNUM', 'RN_Status', 'LPN_Status', 'CNA_Status']].head())

# --- Summary of Staffing Issues ---
print("\n--- Summary of Staffing Issues by Nurse Type ---")
rn_summary = final_df['RN_Status'].value_counts()
lpn_summary = final_df['LPN_Status'].value_counts()
cna_summary = final_df['CNA_Status'].value_counts()

summary_data = {
    'RN': rn_summary,
    'LPN': lpn_summary,
    'CNA': cna_summary
}
staffing_summary_df = pd.DataFrame(summary_data).fillna(0).astype(int)
print(staffing_summary_df)


Final Comparison Data Head (Original vs. Optimized Hours):
  PROVNUM  Original_Total_Hours  Optimized_Total_Hours  Hours_Saved
0  015009              17117.16               14235.50      2881.66
1  015010              24236.25               24015.75       220.50
2  015012              16566.37               13788.20      2778.17
3  015014               9197.82                7895.20      1302.62
4  015015              27460.89               29067.40     -1606.51

--- EVALUATION SUMMARY ---
Original Compliance Rate: 4.78% of facilities were compliant.
Optimized Compliance Rate: 100.00% of facilities are now compliant.
Total Hours Before Optimization: 369,042,003
Total Hours After Optimization: 395,235,281
Total Hours Saved (Net Change): -26,193,278
Percentage of Hours Saved (Net Change): -7.10%

--- Detailed Staffing Status Analysis ---

Detailed status for each facility (first 5 rows):
  PROVNUM     RN_Status    LPN_Status   CNA_Status
0  015009   Overstaffed  Understaffed  Overstaffe

In [8]:
# Block 4.1: Save Final Data for Front-End Application

# Define the file name for the final results table
output_file_name = 'final_results.csv'

# Save the DataFrame to a CSV file.
# index=False ensures the DataFrame index (row numbers) is not included in the CSV.
try:
    final_df.to_csv(output_file_name, index=False)
    print(f"\nSuccessfully saved the final results data to: {output_file_name}")
    print("This file can now be used by the Streamlit application (app.py).")
except NameError:
    print("\nERROR: 'final_df' not found in memory. Please ensure Block 4 was run successfully.")
except Exception as e:
    print(f"\nAn error occurred while saving the file: {e}")


Successfully saved the final results data to: final_results.csv
This file can now be used by the Streamlit application (app.py).


In [5]:
# Block 5: Visualization - Chart Generation (The Full Run)

# --- Visualizing Before vs. After Optimization ---
chart_dir = 'facility_charts'
os.makedirs(chart_dir, exist_ok=True)

# WARNING: This process generates 14,551 files and can be slow if security software is active.
print(f"\n--- Generating and saving {len(final_df)} facility charts... ---")

# Loop through the ENTIRE dataset (no .head() limit)
for index, row in final_df.iterrows():
    facility_id = str(row['PROVNUM'])
    nurse_types = ['RN', 'LPN', 'CNA']
    
    # Extract Original and Optimized Hours
    original_hours = [row['Original_RN_Hours'], row['Original_LPN_Hours'], row['Original_CNA_Hours']]
    optimized_hours = [row['Optimized_RN_Hours'], row['Optimized_LPN_Hours'], row['Optimized_CNA_Hours']]
    
    x = np.arange(len(nurse_types))
    width = 0.35
    
    # Create the plot
    fig, ax = plt.subplots(figsize=(10, 6))
    rects1 = ax.bar(x - width/2, original_hours, width, label='Original Hours', color='skyblue')
    rects2 = ax.bar(x + width/2, optimized_hours, width, label='Optimized Hours', color='coral')
    
    # Set labels and title
    ax.set_ylabel('Total Hours')
    ax.set_title(f'Staffing Hours Comparison for Facility: {facility_id}')
    ax.set_xticks(x)
    ax.set_xticklabels(nurse_types)
    ax.legend(loc='upper right') # Explicit loc='upper right' avoids the slow 'best' calculation
    
    # Add data labels on bars
    ax.bar_label(rects1, padding=3, fmt='%.0f')
    ax.bar_label(rects2, padding=3, fmt='%.0f')
    
    fig.tight_layout()
    
    # Save the figure to the local file system
    plt.savefig(os.path.join(chart_dir, f'{facility_id}_comparison.png'))
    plt.close(fig) # Essential to free up memory after each plot is saved

print(f"--- All charts have been saved to the '{chart_dir}' folder. ---")


--- Generating and saving 14551 facility charts... ---
--- All charts have been saved to the 'facility_charts' folder. ---


In [9]:
# Block 6a: Minimal Setup for Chart Viewer

# Import necessary libraries
import os
from PIL import Image 

# Define the directory where the charts were saved.
# Note: You specified the absolute path, but the relative path 'facility_charts' 
# should still work if your notebook is in C:\project\OR 1\.
chart_dir = 'facility_charts' 

print(f"Chart directory defined as: {chart_dir}")
print("Ready to run the Interactive Chart Viewer.")

Chart directory defined as: facility_charts
Ready to run the Interactive Chart Viewer.


In [10]:
# Block 6: User Interaction - Interactive Chart Viewer (Instant Demo)

print("\n--- Interactive Chart Viewer Activated ---")

# The chart_dir variable is now available from Block 6a

while True:
    print("\nTo view a specific chart, enter the facility ID (PROVNUM).")
    user_input = input("Enter facility ID (or type 'exit' to quit): ").strip()

    if user_input.lower() in ['exit', 'quit']:
        print("Interactive viewer closed.")
        break
    
    # Construct the path to the chart image using the input PROVNUM
    chart_path = os.path.join(chart_dir, f'{user_input}_comparison.png')

    if os.path.exists(chart_path):
        try:
            # Open and show the chart using the OS default viewer
            img = Image.open(chart_path)
            img.show()
            print(f"Opening chart for {user_input}...")
        except Exception as e:
            print(f"Error: Could not open image file. Reason: {e}")
    else:
        print(f"Error: A chart for facility ID '{user_input}' was not found. Please ensure the PROVNUM is correct.")


--- Interactive Chart Viewer Activated ---

To view a specific chart, enter the facility ID (PROVNUM).
Opening chart for 335711...

To view a specific chart, enter the facility ID (PROVNUM).
Error: A chart for facility ID '' was not found. Please ensure the PROVNUM is correct.

To view a specific chart, enter the facility ID (PROVNUM).
Error: A chart for facility ID '' was not found. Please ensure the PROVNUM is correct.

To view a specific chart, enter the facility ID (PROVNUM).
Interactive viewer closed.


In [11]:
# Block 6: User Interaction - Interactive Chart Viewer (Instant Demo)

# NOTE: This requires Block 5 to have been run successfully and the 'Image' object 
# from the PIL library to be available (imported in Block 1).

print("\n--- Interactive Chart Viewer Activated ---")

# The path must be defined here as it was in Block 5
chart_dir = 'facility_charts' 

while True:
    print("\nTo view a specific chart, enter the facility ID (PROVNUM).")
    user_input = input("Enter facility ID (or type 'exit' to quit): ").strip()

    if user_input.lower() in ['exit', 'quit']:
        print("Interactive viewer closed.")
        break
    
    # Construct the path to the chart image using the input PROVNUM
    chart_path = os.path.join(chart_dir, f'{user_input}_comparison.png')

    if os.path.exists(chart_path):
        try:
            # Open and show the chart using the OS default viewer
            img = Image.open(chart_path)
            img.show()
            print(f"Opening chart for {user_input}...")
        except Exception as e:
            print(f"Error: Could not open image file. Reason: {e}")
    else:
        print(f"Error: A chart for facility ID '{user_input}' was not found. Please ensure the PROVNUM is correct.")


--- Interactive Chart Viewer Activated ---

To view a specific chart, enter the facility ID (PROVNUM).
Opening chart for 745055...

To view a specific chart, enter the facility ID (PROVNUM).
Interactive viewer closed.
